In [0]:
%load_ext autoreload
%autoreload 2
# Enables autoreload; learn more at https://docs.databricks.com/en/files/workspace-modules.html#autoreload-for-python-modules
# To disable autoreload; run %autoreload 0

# LexAI One-Click Databricks Runner

Run this notebook to initialize the notebook-06 engine and optionally expose API/UI from the same cluster session.

## What this notebook does
- Validates paths and cluster context
- Installs missing app dependencies (optional)
- Initializes notebook 06 runtime via adapter
- Runs smoke-test legal queries
- Optionally starts FastAPI and Streamlit

## Prerequisite
- `04_generate_embedding_Test.ipynb` and `06_High-precision_QA_Legal_Reasoning_Engine.ipynb` logic should be available in this repo.
- Embedding Delta data should exist in the configured Volume/table.


In [0]:
REPO_DIR_OVERRIDE = "/Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform"

In [0]:
# CELL 1: Runtime Flags (edit if needed)
AUTO_INSTALL_MISSING = True
RUN_SMOKE_TEST = True
START_FASTAPI = True
START_STREAMLIT = True

# Smoke test queries
SMOKE_TEST_QUERIES = [
    "Penalty for not wearing helmet in short within 120 words",
    "What does section 129 say in detail within 180 words",
]

# Server ports
FASTAPI_PORT = 8000
STREAMLIT_PORT = 8501

# Optional: override if repo location differs
REPO_DIR_OVERRIDE = ""

print("[CELL 1] Flags loaded")
print({
    "AUTO_INSTALL_MISSING": AUTO_INSTALL_MISSING,
    "RUN_SMOKE_TEST": RUN_SMOKE_TEST,
    "START_FASTAPI": START_FASTAPI,
    "START_STREAMLIT": START_STREAMLIT,
    "FASTAPI_PORT": FASTAPI_PORT,
    "STREAMLIT_PORT": STREAMLIT_PORT,
})


[CELL 1] Flags loaded
{'AUTO_INSTALL_MISSING': True, 'RUN_SMOKE_TEST': True, 'START_FASTAPI': True, 'START_STREAMLIT': True, 'FASTAPI_PORT': 8000, 'STREAMLIT_PORT': 8501}


In [0]:
# CELL 2: Resolve repo path and set working directory
import os
import sys
from pathlib import Path
from datetime import datetime


def log(msg: str):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def _repo_has_app_files(repo_dir: Path) -> bool:
    return (repo_dir / "apps" / "fastapi_app.py").exists() and (repo_dir / "apps" / "lexai06_notebook_adapter.py").exists()


def _safe_repo_scan(root: Path):
    skip_dirs = {"__pycache__", ".git", ".ipynb_checkpoints"}

    def _onerror(_err):
        return None

    for dirpath, dirnames, _filenames in os.walk(root, topdown=True, onerror=_onerror):
        dirnames[:] = [d for d in dirnames if d not in skip_dirs]
        repo_dir = Path(dirpath)
        try:
            if _repo_has_app_files(repo_dir):
                return repo_dir
        except Exception:
            continue
    return None


def _repo_from_notebook_context():
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        p = ctx.notebookPath().get()  # e.g. /Users/<email>/lexai-legal-rag-platform/notebooks/07_one_click_lexai_runner
        if not p:
            return None
        pp = Path(p)
        # expected: /Users/.../<repo>/notebooks/<nb>
        if len(pp.parts) >= 3:
            repo_guess = pp.parent.parent
            # filesystem view often under /Workspace
            fs_guess = Path("/Workspace") / Path(*repo_guess.parts[1:]) if str(repo_guess).startswith('/') else Path('/Workspace') / repo_guess
            return fs_guess
    except Exception:
        return None
    return None


def resolve_repo_dir() -> Path:
    # 1) explicit override
    if REPO_DIR_OVERRIDE and str(REPO_DIR_OVERRIDE).strip():
        p = Path(REPO_DIR_OVERRIDE.strip())
        if p.exists() and _repo_has_app_files(p):
            return p

    # 2) current working dir and parents
    cwd = Path(os.getcwd()).resolve()
    for candidate in [cwd] + list(cwd.parents):
        try:
            if _repo_has_app_files(candidate):
                return candidate
        except Exception:
            continue

    # 3) notebook context-derived guess
    ctx_guess = _repo_from_notebook_context()
    if ctx_guess is not None:
        try:
            if ctx_guess.exists() and _repo_has_app_files(ctx_guess):
                return ctx_guess
        except Exception:
            pass

    # 4) workspace scan
    for root in [Path('/Workspace/Repos'), Path('/Workspace/Users'), Path('/Workspace')]:
        if not root.exists():
            continue
        hit = _safe_repo_scan(root)
        if hit is not None:
            return hit

    raise FileNotFoundError(
        "Could not locate repo root containing apps/fastapi_app.py and apps/lexai06_notebook_adapter.py. "
        "Set REPO_DIR_OVERRIDE to your repo path (example: /Workspace/Users/<email>/lexai-legal-rag-platform)."
    )


REPO_DIR = resolve_repo_dir()
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

log(f"Repo root: {REPO_DIR}")
log(f"Working directory: {Path.cwd()}")
print("[CELL 2] OK")


[13:58:26] Repo root: /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform
[13:58:26] Working directory: /Workspace/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform
[CELL 2] OK


In [0]:
# CELL 3: Optional dependency install (idempotent)
import importlib.metadata as ilm
import subprocess

REQ_FILE = Path("apps/requirements.txt")
if not REQ_FILE.exists():
    raise FileNotFoundError(f"Missing requirements file: {REQ_FILE}")

# App + notebook-06 runtime dependencies
required_pkgs = [
    "fastapi",
    "uvicorn",
    "streamlit",
    "requests",
    "pydantic",
    "sentence-transformers",
    "transformers",
    "accelerate",
    "mlflow",
    "databricks-sdk",
]
missing = []
for pkg in required_pkgs:
    pkg_name = pkg.split(">")[0].split("=")[0].strip()
    try:
        ilm.version(pkg_name)
    except Exception:
        missing.append(pkg)

print("[CELL 3] Missing packages:", missing)

# Always upgrade typing_extensions to ensure >=4.6.0 (required for TypeIs in transformers/sentence-transformers)
typing_ext_current = "unknown"
try:
    typing_ext_current = ilm.version("typing_extensions")
except:
    pass

print(f"[CELL 3] Current typing_extensions version: {typing_ext_current}")
print("[CELL 3] Force-upgrading typing_extensions to >=4.6.0...")
cmd = [sys.executable, "-m", "pip", "install", "-q", "--upgrade", "typing_extensions>=4.6.0"]
subprocess.check_call(cmd)

if missing and AUTO_INSTALL_MISSING:
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + missing
    print("[CELL 3] Installing missing packages...")
    subprocess.check_call(cmd)
    print("[CELL 3] Installation completed")

print("[CELL 3] Python restart required for typing_extensions upgrade to take effect")
dbutils.library.restartPython()

[CELL 3] Missing packages: []
[CELL 3] Current typing_extensions version: 4.15.0
[CELL 3] Force-upgrading typing_extensions to >=4.6.0...
[CELL 3] Python restart required for typing_extensions upgrade to take effect



[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [0]:
# CELL 4: Validate Spark and Databricks context
from pyspark.sql import SparkSession

spark = SparkSession.getActiveSession() or SparkSession.builder.getOrCreate()
print("[CELL 4] Spark session ready:", bool(spark))

cluster_id = "unknown"
org_id = "unknown"
workspace_url = "unknown"

try:
    cluster_id = spark.conf.get("spark.databricks.clusterUsageTags.clusterId")
except Exception:
    pass

try:
    org_id = spark.conf.get("spark.databricks.clusterUsageTags.orgId")
except Exception:
    pass

try:
    workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
except Exception:
    pass

print("[CELL 4] cluster_id:", cluster_id)
print("[CELL 4] org_id:", org_id)
print("[CELL 4] workspace_url:", workspace_url)


[CELL 4] Spark session ready: True


2026-03-01 13:59:27,469 22501 ERROR _handle_rpc_error GRPC Error received
Traceback (most recent call last):
  File "/databricks/python/lib/python3.10/site-packages/pyspark/sql/connect/client/core.py", line 1724, in config
    resp = self._stub.Config(req, metadata=self.metadata())
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 277, in __call__
    response, ignored_call = self._with_call(
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 332, in _with_call
    return call.result(), call
  File "/databricks/python/lib/python3.10/site-packages/grpc/_channel.py", line 439, in result
    raise self
  File "/databricks/python/lib/python3.10/site-packages/grpc/_interceptor.py", line 315, in continuation
    response, call = self._thunk(new_method).with_call(
  File "/databricks/python/lib/python3.10/site-packages/grpc/_channel.py", line 1193, in with_call
    return _end_unary_response_blocking(state, call, True, None)
 

[CELL 4] cluster_id: 0301-061338-cb67io08-v2n
[CELL 4] org_id: unknown
[CELL 4] workspace_url: dbc-afb2e98d-d930.cloud.databricks.com


In [0]:
# CELL 5: Initialize notebook-06 engine through adapter (deterministic path)
from pathlib import Path
import importlib
import os
import tempfile
import base64
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.workspace import ExportFormat
import apps.lexai06_notebook_adapter as _adapter

importlib.reload(_adapter)
NotebookEngine = _adapter.NotebookEngine

# Export the notebook using Databricks SDK (dbutils.workspace.export doesn't work in this environment)
print("[CELL 5] Exporting notebook...")
w = WorkspaceClient()
workspace_path = "/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/notebooks/06_High-precision_QA_Legal_Reasoning_Engine"

try:
    exported = w.workspace.export(path=workspace_path, format=ExportFormat.JUPYTER)
    # Decode base64 content
    decoded_content = base64.b64decode(exported.content).decode('utf-8')
    temp_dir = Path(tempfile.gettempdir())
    temp_notebook = temp_dir / "lexai06_exported_from_workspace.ipynb"
    temp_notebook.write_text(decoded_content, encoding='utf-8')
    print(f"[CELL 5] Exported to: {temp_notebook} ({temp_notebook.stat().st_size:,} bytes)")
except Exception as e:
    print(f"[CELL 5] Export failed: {e}")
    # Try the snapshot as fallback
    try:
        exported = w.workspace.export(
            path="/Users/kumarangad54706@gmail.com/lexai-legal-rag-platform/apps/notebook_06_snapshot",
            format=ExportFormat.JUPYTER
        )
        decoded_content = base64.b64decode(exported.content).decode('utf-8')
        temp_notebook.write_text(decoded_content, encoding='utf-8')
        print(f"[CELL 5] Using snapshot fallback: {temp_notebook}")
    except Exception as e2:
        raise RuntimeError(f"Could not export notebook: {e2}")

# Initialize the engine with the temp file
os.environ["LEXAI06_NOTEBOOK_PATH"] = str(temp_notebook)
engine = NotebookEngine(notebook_path=temp_notebook)
status = engine.initialize()

print("[CELL 5] Engine initialized")
for k, v in status.items():
    print(f"  - {k}: {v}")

if not status.get("ready"):
    raise RuntimeError(f"Engine failed to initialize: {status}")

[CELL 5] Exporting notebook...
[CELL 5] Exported to: /tmp/lexai06_exported_from_workspace.ipynb (96,938 bytes)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[14:00:05] Embedding model ready: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[14:00:06] Using endpoint backend: databricks-meta-llama-3-3-70b-instruct
[14:00:13] Loaded lexical artifacts from Delta for signature=312b53f785d939bab1c3.
--- Runtime Status ---
Data signature: 312b53f785d939bab1c3
Embedding source: /Volumes/workspace/legal_data/vector_db_test/legal_embeddings_delta
Records loaded: 5194
Embedding dim: 384
Avg doc len: 183.25
Vocabulary size: 13196
Lexical source: delta_artifact
Embedder: sentence-transformers/all-MiniLM-L6-v2
Reranker: cross-encoder/ms-marco-MiniLM-L-6-v2
LLM backend: endpoint (databricks-meta-llama-3-3-70b-instruct)
[CELL 5] Engine initialized
  - ready: True
  - error: 
  - notebook_path: /tmp/lexai06_exported_from_workspace.ipynb
  - resolved_notebook_path: /tmp/lexai06_exported_from_workspace.ipynb
  - exec_cell_numbers: [2, 3, 4, 5, 6, 7, 8, 9]
  - init_ms: 10462.55
  - data_signature: 312b53f785d939bab1c3
  - records_loaded: 5194
  - embedder: sentence-transformers/all-MiniLM-L6-v2
  - reranker: cross-encoder/ms-marco-MiniLM-L-

In [0]:
# CELL 6: Smoke test queries (citation-rich output check)
if RUN_SMOKE_TEST:
    print("[CELL 6] Running smoke tests...")
    for idx, q in enumerate(SMOKE_TEST_QUERIES, start=1):
        print("=" * 90)
        print(f"[{idx}] Query: {q}")
        out = engine.answer_query(q)
        print("Mode:", out.get("mode"))
        print("Source:", out.get("source"))
        print("Confidence:", out.get("confidence"))
        print("Sections:", out.get("sections", []))
        print("Citations:", out.get("citations", [])[:5])
        print("Latency:", out.get("latency_ms", {}))
        print("Answer:")
        print(out.get("answer", ""))
    print("[CELL 6] Smoke tests done")
else:
    print("[CELL 6] RUN_SMOKE_TEST=False -> skipped")


[CELL 6] Running smoke tests...
[1] Query: Penalty for not wearing helmet in short within 120 words
Mode: rule_based
Source: traffic_rules
Confidence: top_score=0.617, avg_lex=0.732, coverage=0.444, normative_hits=5, judgment_hits=0, candidates=2512, shortlist=140, route=statute_strict
Sections: ['177', '178', '179', '180', '181', '182', '183', '184', '186', '189', '190', '191', '129']
Citations: ['Motor Vehicles Act 1988 - Section Chapter V', 'Motor Vehicles Act 1988 - Section Chapter VII', 'Motor Vehicle Ammendment Act 2019 - Section Chapter XI', 'Motor Vehicles Act 1988 - Section Chapter VI', 'Central Motor Vehicle Rules 1989 - Section Chapter VI']
Latency: {'candidate_fetch_ms': 5.1, 'routing_ms': 80.63, 'lexical_ms': 135.93, 'embed_ms': 69.7, 'dense_ms': 6.86, 'rrf_ms': 0.29, 'rerank_ms': 1176.0, 'retrieve_total_ms': 1480.79, 'generation_ms': 0.0, 'total_ms': 1482.57}
Answer:
Law:
Section 129 of the Motor Vehicles Act requires riders to wear protective headgear while riding two-wh

In [0]:
# CELL 7: Start FastAPI server (background thread)
import threading
import uvicorn

FASTAPI_SERVER = globals().get("FASTAPI_SERVER")
FASTAPI_THREAD = globals().get("FASTAPI_THREAD")

if START_FASTAPI:
    if FASTAPI_THREAD is not None and FASTAPI_THREAD.is_alive():
        print(f"[CELL 7] FastAPI already running on port {FASTAPI_PORT}")
    else:
        from apps.fastapi_app import app

        config = uvicorn.Config(app, host="0.0.0.0", port=int(FASTAPI_PORT), log_level="info")
        FASTAPI_SERVER = uvicorn.Server(config)
        FASTAPI_THREAD = threading.Thread(target=FASTAPI_SERVER.run, daemon=True)
        FASTAPI_THREAD.start()
        globals()["FASTAPI_SERVER"] = FASTAPI_SERVER
        globals()["FASTAPI_THREAD"] = FASTAPI_THREAD
        print(f"[CELL 7] FastAPI started on 0.0.0.0:{FASTAPI_PORT}")

    print("[CELL 7] Local health URL:", f"http://127.0.0.1:{FASTAPI_PORT}/health")

    try:
        if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
            proxy_url = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}/{FASTAPI_PORT}/health"
            print("[CELL 7] Driver proxy URL:", proxy_url)
    except Exception:
        pass
else:
    print("[CELL 7] START_FASTAPI=False -> skipped")


[CELL 7] FastAPI started on 0.0.0.0:8000
[CELL 7] Local health URL: http://127.0.0.1:8000/health


In [0]:
# CELL 8: Optional FastAPI smoke call
import requests

if START_FASTAPI:
    try:
        h = requests.get(f"http://127.0.0.1:{FASTAPI_PORT}/health", timeout=30)
        print("[CELL 8] /health status:", h.status_code)
        print(h.json())

        payload = {
            "query": "What is the penalty for not wearing a helmet?",
            "style": "short",
            "word_limit": 120,
        }
        r = requests.post(f"http://127.0.0.1:{FASTAPI_PORT}/v1/legal/answer", json=payload, timeout=180)
        print("[CELL 8] /v1/legal/answer status:", r.status_code)
        print("[CELL 8] answer preview:", r.json().get("answer", "")[:500])
    except Exception as e:
        print("[CELL 8] API call failed:", e)
else:
    print("[CELL 8] START_FASTAPI=False -> skipped")


INFO:     Started server process [22501]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


[CELL 8] API call failed: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError('<urllib3.connection.HTTPConnection object at 0x7f0463d17a00>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [0]:
# CELL 9: Optional Streamlit start (blocking cell)
import os
import subprocess

if START_STREAMLIT:
    os.environ["LEXAI_API_BASE_URL"] = f"http://127.0.0.1:{FASTAPI_PORT}"
    cmd = [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        "apps/streamlit_app.py",
        "--server.port",
        str(STREAMLIT_PORT),
        "--server.address",
        "0.0.0.0",
    ]
    print("[CELL 9] Starting Streamlit:", " ".join(cmd))
    if workspace_url != "unknown" and org_id != "unknown" and cluster_id != "unknown":
        ui_url = f"https://{workspace_url}/driver-proxy/o/{org_id}/{cluster_id}/{STREAMLIT_PORT}/"
        print("[CELL 9] Streamlit URL:", ui_url)
    print("[CELL 9] This cell is blocking while Streamlit is running.")
    subprocess.call(cmd)
else:
    print("[CELL 9] START_STREAMLIT=False -> skipped")


[CELL 9] Starting Streamlit: /local_disk0/.ephemeral_nfs/envs/pythonEnv-4a17e6d4-ef23-407a-b783-18b3ea2458c4/bin/python -m streamlit run apps/streamlit_app.py --server.port 8501 --server.address 0.0.0.0
[CELL 9] This cell is blocking while Streamlit is running.



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.210.126:8501
  External URL: http://3.145.247.26:8501



In [0]:
# CELL 10: Stop helper (run when needed)
if "FASTAPI_SERVER" in globals() and globals().get("FASTAPI_SERVER") is not None:
    globals()["FASTAPI_SERVER"].should_exit = True
    print("[CELL 10] FastAPI stop requested")
else:
    print("[CELL 10] FastAPI was not running")
